# Zenith Rigorous Benchmark Suite

**Purpose:** Mathematically rigorous performance benchmarking with honest reporting.

**Methodology:**
- 30 measured runs per test (statistically significant)
- 5 warmup runs (discarded)
- Report: mean ± std, 95% CI
- Fixed random seeds for reproducibility

**Models Tested:**
1. BERT-base (110M params) - NLP Encoder
2. ResNet-50 (25.5M params) - Computer Vision
3. TinyLlama 1.1B - LLM Decoder

**Hardware:** NVIDIA Tesla T4 (Google Colab)

---

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q transformers torch torchvision numpy scipy pandas matplotlib
!pip install -q pyzenith  # Official Zenith package

In [ ]:
# Record environment for reproducibility
import torch
import numpy as np
import sys
import platform
from datetime import datetime

print("=" * 60)
print("BENCHMARK ENVIRONMENT")
print("=" * 60)
print(f"Date: {datetime.now().isoformat()}")
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Platform: {platform.platform()}")

if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.2f} GB")
    print(f"Compute Capability: {props.major}.{props.minor}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    raise RuntimeError("GPU not available! This benchmark requires a GPU.")

# Try to import Zenith
try:
    import zenith
    print(f"\nZenith: {zenith.__version__}")
    ZENITH_AVAILABLE = True
except ImportError:
    print("\nWARNING: Zenith not available. Will benchmark torch.compile only.")
    ZENITH_AVAILABLE = False

## 2. Benchmark Infrastructure

In [ ]:
import gc
import time
from typing import Dict, List, Callable, Any, Optional
from dataclasses import dataclass, field
from scipy import stats
import pandas as pd

# Configuration
WARMUP_RUNS = 5
MEASURED_RUNS = 30
RANDOM_SEED = 42

@dataclass
class BenchmarkResult:
    """Stores benchmark results with statistical analysis."""
    name: str
    backend: str
    timings_ms: List[float] = field(default_factory=list)
    peak_memory_gb: float = 0.0
    throughput_samples_per_sec: float = 0.0
    accuracy_mse: Optional[float] = None
    
    @property
    def mean_ms(self) -> float:
        return np.mean(self.timings_ms) if self.timings_ms else 0.0
    
    @property
    def std_ms(self) -> float:
        return np.std(self.timings_ms, ddof=1) if len(self.timings_ms) > 1 else 0.0
    
    @property
    def p50_ms(self) -> float:
        return np.percentile(self.timings_ms, 50) if self.timings_ms else 0.0
    
    @property
    def p95_ms(self) -> float:
        return np.percentile(self.timings_ms, 95) if self.timings_ms else 0.0
    
    @property
    def p99_ms(self) -> float:
        return np.percentile(self.timings_ms, 99) if self.timings_ms else 0.0
    
    @property
    def ci_95(self) -> tuple:
        """95% confidence interval."""
        if len(self.timings_ms) < 2:
            return (0.0, 0.0)
        ci = stats.t.interval(
            0.95, 
            len(self.timings_ms) - 1,
            loc=self.mean_ms,
            scale=stats.sem(self.timings_ms)
        )
        return (ci[0], ci[1])


def clean_memory():
    """Force garbage collection and clear CUDA cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


def set_seed(seed: int = RANDOM_SEED):
    """Set all random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def benchmark_function(
    name: str,
    backend: str,
    func: Callable,
    inputs: Any,
    warmup: int = WARMUP_RUNS,
    runs: int = MEASURED_RUNS,
) -> BenchmarkResult:
    """
    Run benchmark with proper warmup and statistical measurement.
    
    Args:
        name: Model name
        backend: Backend name (pytorch, zenith, etc.)
        func: Function to benchmark (takes inputs, returns output)
        inputs: Input data for the function
        warmup: Number of warmup runs (discarded)
        runs: Number of measured runs
    
    Returns:
        BenchmarkResult with statistics
    """
    result = BenchmarkResult(name=name, backend=backend)
    
    clean_memory()
    set_seed()
    
    # Warmup (discarded)
    print(f"  Warming up ({warmup} runs)...", end=" ", flush=True)
    for _ in range(warmup):
        _ = func(inputs)
        torch.cuda.synchronize()
    print("done")
    
    # Reset memory stats after warmup
    torch.cuda.reset_peak_memory_stats()
    
    # Measured runs
    print(f"  Measuring ({runs} runs)...", end=" ", flush=True)
    for i in range(runs):
        torch.cuda.synchronize()
        start = time.perf_counter()
        
        _ = func(inputs)
        
        torch.cuda.synchronize()
        end = time.perf_counter()
        
        result.timings_ms.append((end - start) * 1000)
    print("done")
    
    # Record peak memory
    result.peak_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    
    return result


def compare_outputs(output1: torch.Tensor, output2: torch.Tensor) -> float:
    """Calculate MSE between two outputs for accuracy verification."""
    with torch.no_grad():
        if output1.shape != output2.shape:
            return float('inf')
        mse = torch.mean((output1.float() - output2.float()) ** 2).item()
    return mse


print("Benchmark infrastructure initialized.")
print(f"  Warmup runs: {WARMUP_RUNS}")
print(f"  Measured runs: {MEASURED_RUNS}")
print(f"  Random seed: {RANDOM_SEED}")

## 3. Model Benchmarks

### 3.1 BERT-base (NLP Encoder)

In [ ]:
from transformers import BertModel, BertTokenizer

print("=" * 60)
print("BERT-BASE BENCHMARK")
print("=" * 60)

# Load model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model_bert = BertModel.from_pretrained('bert-base-uncased').cuda().eval()

# Prepare input (batch_size=8, seq_len=128)
batch_size = 8
seq_len = 128
dummy_text = "The quick brown fox jumps over the lazy dog. " * 10
inputs_bert = tokenizer(
    [dummy_text] * batch_size,
    padding='max_length',
    max_length=seq_len,
    truncation=True,
    return_tensors='pt'
).to('cuda')

print(f"\nInput shape: batch={batch_size}, seq_len={seq_len}")
print(f"Model params: {sum(p.numel() for p in model_bert.parameters()) / 1e6:.1f}M")

# PyTorch baseline
print("\n[1/3] PyTorch Baseline (no compilation)")
with torch.no_grad():
    bert_pytorch = benchmark_function(
        name="BERT-base",
        backend="pytorch",
        func=lambda x: model_bert(**x).last_hidden_state,
        inputs=inputs_bert
    )

# torch.compile (inductor)
print("\n[2/3] torch.compile (inductor backend)")
clean_memory()
model_bert_compiled = torch.compile(model_bert, backend='inductor')
with torch.no_grad():
    bert_inductor = benchmark_function(
        name="BERT-base",
        backend="inductor",
        func=lambda x: model_bert_compiled(**x).last_hidden_state,
        inputs=inputs_bert
    )

# Zenith backend (if available)
if ZENITH_AVAILABLE:
    print("\n[3/3] Zenith Backend")
    clean_memory()
    model_bert_fresh = BertModel.from_pretrained('bert-base-uncased').cuda().eval()
    model_bert_zenith = torch.compile(model_bert_fresh, backend='zenith')
    with torch.no_grad():
        bert_zenith = benchmark_function(
            name="BERT-base",
            backend="zenith",
            func=lambda x: model_bert_zenith(**x).last_hidden_state,
            inputs=inputs_bert
        )
else:
    print("\n[3/3] Zenith Backend - SKIPPED (not available)")
    bert_zenith = None

# Accuracy verification
print("\nAccuracy Verification:")
with torch.no_grad():
    output_pytorch = model_bert(**inputs_bert).last_hidden_state
    output_inductor = model_bert_compiled(**inputs_bert).last_hidden_state
    mse_inductor = compare_outputs(output_pytorch, output_inductor)
    print(f"  PyTorch vs Inductor MSE: {mse_inductor:.2e}")
    bert_inductor.accuracy_mse = mse_inductor
    
    if ZENITH_AVAILABLE and bert_zenith:
        output_zenith = model_bert_zenith(**inputs_bert).last_hidden_state
        mse_zenith = compare_outputs(output_pytorch, output_zenith)
        print(f"  PyTorch vs Zenith MSE: {mse_zenith:.2e}")
        bert_zenith.accuracy_mse = mse_zenith

# Store results
bert_results = [bert_pytorch, bert_inductor]
if bert_zenith:
    bert_results.append(bert_zenith)

# Cleanup
del model_bert, model_bert_compiled
if ZENITH_AVAILABLE:
    del model_bert_fresh, model_bert_zenith
clean_memory()

### 3.2 ResNet-50 (Computer Vision)

In [ ]:
import torchvision.models as models

print("=" * 60)
print("RESNET-50 BENCHMARK")
print("=" * 60)

# Load model
model_resnet = models.resnet50(pretrained=True).cuda().eval()

# Prepare input (batch_size=32, 224x224)
batch_size = 32
inputs_resnet = torch.randn(batch_size, 3, 224, 224).cuda()

print(f"\nInput shape: {inputs_resnet.shape}")
print(f"Model params: {sum(p.numel() for p in model_resnet.parameters()) / 1e6:.1f}M")

# PyTorch baseline
print("\n[1/3] PyTorch Baseline (no compilation)")
with torch.no_grad():
    resnet_pytorch = benchmark_function(
        name="ResNet-50",
        backend="pytorch",
        func=lambda x: model_resnet(x),
        inputs=inputs_resnet
    )

# torch.compile (inductor)
print("\n[2/3] torch.compile (inductor backend)")
clean_memory()
model_resnet_compiled = torch.compile(model_resnet, backend='inductor')
with torch.no_grad():
    resnet_inductor = benchmark_function(
        name="ResNet-50",
        backend="inductor",
        func=lambda x: model_resnet_compiled(x),
        inputs=inputs_resnet
    )

# Zenith backend
if ZENITH_AVAILABLE:
    print("\n[3/3] Zenith Backend")
    clean_memory()
    model_resnet_fresh = models.resnet50(pretrained=True).cuda().eval()
    model_resnet_zenith = torch.compile(model_resnet_fresh, backend='zenith')
    with torch.no_grad():
        resnet_zenith = benchmark_function(
            name="ResNet-50",
            backend="zenith",
            func=lambda x: model_resnet_zenith(x),
            inputs=inputs_resnet
        )
else:
    print("\n[3/3] Zenith Backend - SKIPPED")
    resnet_zenith = None

# Accuracy verification
print("\nAccuracy Verification:")
with torch.no_grad():
    output_pytorch = model_resnet(inputs_resnet)
    output_inductor = model_resnet_compiled(inputs_resnet)
    mse_inductor = compare_outputs(output_pytorch, output_inductor)
    print(f"  PyTorch vs Inductor MSE: {mse_inductor:.2e}")
    resnet_inductor.accuracy_mse = mse_inductor
    
    if ZENITH_AVAILABLE and resnet_zenith:
        output_zenith = model_resnet_zenith(inputs_resnet)
        mse_zenith = compare_outputs(output_pytorch, output_zenith)
        print(f"  PyTorch vs Zenith MSE: {mse_zenith:.2e}")
        resnet_zenith.accuracy_mse = mse_zenith

# Store results
resnet_results = [resnet_pytorch, resnet_inductor]
if resnet_zenith:
    resnet_results.append(resnet_zenith)

# Cleanup
del model_resnet, model_resnet_compiled, inputs_resnet
if ZENITH_AVAILABLE:
    del model_resnet_fresh, model_resnet_zenith
clean_memory()

### 3.3 TinyLlama 1.1B (LLM Decoder)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print("=" * 60)
print("TINYLLAMA 1.1B BENCHMARK")
print("=" * 60)

# Load model (use float16 to fit in T4 memory)
tokenizer_llama = AutoTokenizer.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0')
model_llama = AutoModelForCausalLM.from_pretrained(
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    torch_dtype=torch.float16,
    device_map='cuda'
).eval()

# Prepare input (smaller batch for memory)
batch_size = 4
seq_len = 64
inputs_llama = torch.randint(0, tokenizer_llama.vocab_size, (batch_size, seq_len)).cuda()

print(f"\nInput shape: batch={batch_size}, seq_len={seq_len}")
print(f"Model params: {sum(p.numel() for p in model_llama.parameters()) / 1e9:.2f}B")

# PyTorch baseline
print("\n[1/3] PyTorch Baseline (no compilation)")
with torch.no_grad():
    llama_pytorch = benchmark_function(
        name="TinyLlama-1.1B",
        backend="pytorch",
        func=lambda x: model_llama(x).logits,
        inputs=inputs_llama
    )

# torch.compile (inductor)
print("\n[2/3] torch.compile (inductor backend)")
clean_memory()
model_llama_compiled = torch.compile(model_llama, backend='inductor')
with torch.no_grad():
    llama_inductor = benchmark_function(
        name="TinyLlama-1.1B",
        backend="inductor",
        func=lambda x: model_llama_compiled(x).logits,
        inputs=inputs_llama
    )

# Zenith backend
if ZENITH_AVAILABLE:
    print("\n[3/3] Zenith Backend")
    clean_memory()
    model_llama_fresh = AutoModelForCausalLM.from_pretrained(
        'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        torch_dtype=torch.float16,
        device_map='cuda'
    ).eval()
    model_llama_zenith = torch.compile(model_llama_fresh, backend='zenith')
    with torch.no_grad():
        llama_zenith = benchmark_function(
            name="TinyLlama-1.1B",
            backend="zenith",
            func=lambda x: model_llama_zenith(x).logits,
            inputs=inputs_llama
        )
else:
    print("\n[3/3] Zenith Backend - SKIPPED")
    llama_zenith = None

# Accuracy verification
print("\nAccuracy Verification:")
with torch.no_grad():
    output_pytorch = model_llama(inputs_llama).logits
    output_inductor = model_llama_compiled(inputs_llama).logits
    mse_inductor = compare_outputs(output_pytorch, output_inductor)
    print(f"  PyTorch vs Inductor MSE: {mse_inductor:.2e}")
    llama_inductor.accuracy_mse = mse_inductor
    
    if ZENITH_AVAILABLE and llama_zenith:
        output_zenith = model_llama_zenith(inputs_llama).logits
        mse_zenith = compare_outputs(output_pytorch, output_zenith)
        print(f"  PyTorch vs Zenith MSE: {mse_zenith:.2e}")
        llama_zenith.accuracy_mse = mse_zenith

# Store results
llama_results = [llama_pytorch, llama_inductor]
if llama_zenith:
    llama_results.append(llama_zenith)

# Cleanup
del model_llama, model_llama_compiled, inputs_llama
if ZENITH_AVAILABLE:
    del model_llama_fresh, model_llama_zenith
clean_memory()

## 4. Results Analysis

In [ ]:
# Combine all results
all_results = bert_results + resnet_results + llama_results

# Create DataFrame
data = []
for r in all_results:
    ci = r.ci_95
    data.append({
        'Model': r.name,
        'Backend': r.backend,
        'Mean (ms)': f"{r.mean_ms:.2f}",
        'Std (ms)': f"{r.std_ms:.2f}",
        'p50 (ms)': f"{r.p50_ms:.2f}",
        'p95 (ms)': f"{r.p95_ms:.2f}",
        'p99 (ms)': f"{r.p99_ms:.2f}",
        '95% CI': f"[{ci[0]:.2f}, {ci[1]:.2f}]",
        'Peak Mem (GB)': f"{r.peak_memory_gb:.2f}",
        'MSE': f"{r.accuracy_mse:.2e}" if r.accuracy_mse is not None else "N/A"
    })

df = pd.DataFrame(data)

print("\n" + "=" * 80)
print("BENCHMARK RESULTS")
print("=" * 80)
print(df.to_string(index=False))

In [ ]:
# Calculate speedups
print("\n" + "=" * 60)
print("SPEEDUP ANALYSIS")
print("=" * 60)

def calculate_speedup(baseline: BenchmarkResult, optimized: BenchmarkResult) -> dict:
    """Calculate speedup with statistical significance."""
    speedup = (baseline.mean_ms - optimized.mean_ms) / baseline.mean_ms * 100
    
    # Welch's t-test for statistical significance
    t_stat, p_value = stats.ttest_ind(
        baseline.timings_ms, 
        optimized.timings_ms, 
        equal_var=False
    )
    
    return {
        'speedup_pct': speedup,
        'p_value': p_value,
        'significant': p_value < 0.05
    }

models = ['BERT-base', 'ResNet-50', 'TinyLlama-1.1B']
for model_name in models:
    print(f"\n{model_name}:")
    
    # Find results for this model
    model_results = [r for r in all_results if r.name == model_name]
    baseline = next((r for r in model_results if r.backend == 'pytorch'), None)
    
    if baseline is None:
        print("  No baseline found")
        continue
    
    for r in model_results:
        if r.backend == 'pytorch':
            continue
        
        analysis = calculate_speedup(baseline, r)
        significance = "***" if analysis['significant'] else "(not significant)"
        direction = "FASTER" if analysis['speedup_pct'] > 0 else "SLOWER"
        
        print(f"  vs {r.backend}:")
        print(f"    Speedup: {analysis['speedup_pct']:+.2f}% ({direction}) {significance}")
        print(f"    p-value: {analysis['p_value']:.4f}")

## 5. Final Report

In [ ]:
from IPython.display import Markdown, display

# Generate markdown report
report = f"""
# Zenith Benchmark Report

**Date:** {datetime.now().isoformat()}
**Hardware:** NVIDIA Tesla T4
**Methodology:** {MEASURED_RUNS} measured runs, {WARMUP_RUNS} warmup runs

## Summary Table

{df.to_markdown(index=False)}

## Key Findings

| Finding | Value | Verdict |
|---------|-------|--------|
"""

# Add findings for each comparison
for model_name in models:
    model_results = [r for r in all_results if r.name == model_name]
    baseline = next((r for r in model_results if r.backend == 'pytorch'), None)
    zenith_result = next((r for r in model_results if r.backend == 'zenith'), None)
    
    if baseline and zenith_result:
        analysis = calculate_speedup(baseline, zenith_result)
        verdict = "PASS" if analysis['speedup_pct'] > 0 else "NEEDS IMPROVEMENT"
        report += f"| {model_name} vs Zenith | {analysis['speedup_pct']:+.2f}% | {verdict} |\n"

report += """
## Methodology Notes

- All runs used fixed random seed for reproducibility
- Confidence intervals use Student's t-distribution (95%)
- Statistical significance tested with Welch's t-test (p < 0.05)
- MSE calculated against PyTorch eager mode baseline
"""

display(Markdown(report))

In [ ]:
# Save raw results for future analysis
import json

raw_data = {
    'metadata': {
        'date': datetime.now().isoformat(),
        'warmup_runs': WARMUP_RUNS,
        'measured_runs': MEASURED_RUNS,
        'random_seed': RANDOM_SEED,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
        'pytorch_version': torch.__version__,
        'zenith_available': ZENITH_AVAILABLE
    },
    'results': []
}

for r in all_results:
    raw_data['results'].append({
        'name': r.name,
        'backend': r.backend,
        'timings_ms': r.timings_ms,
        'peak_memory_gb': r.peak_memory_gb,
        'accuracy_mse': r.accuracy_mse,
        'statistics': {
            'mean_ms': r.mean_ms,
            'std_ms': r.std_ms,
            'p50_ms': r.p50_ms,
            'p95_ms': r.p95_ms,
            'p99_ms': r.p99_ms,
            'ci_95': r.ci_95
        }
    })

# Save to file
with open('benchmark_results.json', 'w') as f:
    json.dump(raw_data, f, indent=2)

print("Results saved to benchmark_results.json")
print("\nTo download: Files (left panel) > benchmark_results.json > Download")